# DC 제거 순서 수정 후 동일 조건 성능 비교
기존 잠정 0/1 라벨, 선택 구간, 참가자 분할, seed, 모델, 학습률, 임계값 유지. 평균을 리샘플링 전에 제거합니다. 개발 비교이며 독립 외부 검증이 아닙니다. 기존 모델과 자료는 보존합니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import subprocess,sys,json,os
drive.mount('/content/drive')
base=Path('/content/drive/MyDrive/Capdyssey/af_integrated')
repo=Path('/content/capdyssey_centered_comparison')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/dung-ho03/capdyssey.git',str(repo)],check=True)
def run(args):
    subprocess.run([sys.executable,'-u',*args],cwd=repo,check=True,env={**os.environ,'OMP_NUM_THREADS':'2','MKL_NUM_THREADS':'2'})
run(['-m','unittest','discover','-s','tests','-p','test_centered_preprocessing.py'])


In [ ]:
run(['scripts/prepare_af_centered.py','--base',str(base)])


In [ ]:
output=base/'models/provisional_centered_v2'
if not (output/'metrics.json').exists():
    run(['scripts/train_af_integrated.py','--bundle',str(base/'processed/provisional_centered_v2'),'--output',str(output),'--epochs','20','--provisional','--split-from',str(base/'models/provisional_exact01_v1/split.json')])
old=json.loads((base/'models/provisional_exact01_v1/metrics.json').read_text())
new=json.loads((output/'metrics.json').read_text())
comparison={'old':old,'centered':new,'same_windows_and_subject_split':True,'interpretation':'Development comparison; provisional labels unchanged'}
(output/'comparison.json').write_text(json.dumps(comparison,indent=2))
print(json.dumps(comparison,indent=2))
